In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

In [ ]:
df = pd.read_csv("galaxy_users.csv")
df.head(2)

### Q1.

In [ ]:
df_q1 = df.loc[:, "OnlineSecurity":"StreamingMovies"].copy()
df_q1 = df_q1.replace({"Yes": 1, "No": 0})
df_q1.head(1)

In [16]:
# df_q1["OnlineSecurity"].unique()
# df_q1["OnlineBackup"].unique()
# df_q1.unique()
# df_q1.drop_duplicates()
# df_q1.apply(lambda x: [x.unique()]) # 시험버전
df_q1.apply(lambda x: x.unique()) # 최신버전

In [17]:
df_q1_sub = df_q1.loc[df_q1["OnlineSecurity"] != "No internet service", ]

In [ ]:
df_q1_sub.apply(lambda x: x.unique())

In [ ]:
df_q1.loc[df_q1["OnlineSecurity"] == "No internet service", ]

만약에, "No internet service"가 중구난방으로 들어있는 경우 효율적인 처리 방법?

In [ ]:
df_q1_sub = df_q1.replace("No internet service", np.nan).dropna()
df_q1_sub.apply(lambda x: x.unique())

In [27]:
ser_cnt = df_q1_sub.sum(axis = 1)
ser_cnt = ser_cnt.value_counts()
round(ser_cnt[1] / ser_cnt[6], 1)

3.4

### Q2.

In [ ]:
df_q2 = df[["tenure", "MonthlyCharges", "TotalCharges"]].copy()
df_q2["month"] = df_q2["TotalCharges"] // df_q2["MonthlyCharges"]
df_q2.head(2)

In [36]:
df_q2.iloc[:, [0, 1, 3]].corr().round(3) # 0.999, .corr() 기본은 피어슨 상관분석

,tenure,MonthlyCharges,month
tenure,1.000,0.247,0.999
MonthlyCharges,0.247,1.000,0.246
month,0.999,0.246,1.000


In [ ]:
df_corr = df_q2.iloc[:, [0, 1, 3]].corr()
df_corr.replace(1, np.nan).max()

In [47]:
df_corr_m = df_corr.reset_index().melt(id_vars = "index")
df_corr_m.loc[df_corr_m["index"] != df_corr_m["variable"], ]

,index,variable,value
1,MonthlyCharges,tenure,0.246862
2,month,tenure,0.998831
3,tenure,MonthlyCharges,0.246862
5,month,MonthlyCharges,0.246164
6,tenure,month,0.998831
7,MonthlyCharges,month,0.246164


### Q3.

In [ ]:
ls_col_x1 = ["SeniorCitizen", "Partner", "Dependents", "tenure", "MonthlyCharges", "TotalCharges"]
ls_col_x2 = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingMovies", "PaperlessBilling"]
df_q3 = df[["Churn"] + ls_col_x1 + ls_col_x2].copy()
df_q3 = df_q3.replace({"Yes": 1, "No": 0})
df_q3.head(2)

In [ ]:
# .select_dtypes() 메서드가 시험버전에서는 구현이 되어있으나 버그로 동작하지 않음.
# df_q3_obj = df_q3.loc[:, df_q3.dtypes == "object"] # 시험버전
df_q3_obj = df_q3.select_dtypes(include = "object") # 최신버전
df_q3_obj.head(2)

In [ ]:
df_q3_obj.apply(lambda x: x.unique())

In [57]:
df_q3 = df_q3.replace("No internet service", -1)

In [ ]:
df_q3.head()

In [60]:
df_train, df_test = train_test_split(df_q3, train_size = 0.7, random_state = 123)
len(df_train), len(df_test)

(4922, 2110)

In [61]:
model_nor = MinMaxScaler().fit(df_train)
arr_train_nor = model_nor.transform(df_train)
arr_test_nor  = model_nor.transform(df_test)

In [ ]:
arr_train_nor[:1, ]

In [ ]:
model_lr = LogisticRegression(random_state = 123)
model_lr.fit(X = arr_train_nor[:, 1:], # 종속변수 index는 0
             y = arr_train_nor[:, 0])
pred = model_lr.predict(arr_test_nor[:, 1:])
pred[:4]

In [67]:
round(f1_score(y_true = arr_test_nor[:, 0], y_pred = pred), 2)

0.55

### Q. 특정 범주를 제외한 나머지 모든 범주를 지정한 값으로 치환하고자 하는 경우
 ※ "color" 변수의 원소가 "E" 또는 "J"가 아닌 나머지 모든 변수의 범주를 -1로 치환  
 ※ 고수는 .map() + 사용자 정의 함수(or lambda) 로 해결

In [ ]:
df_dia = pd.read_csv("../diamonds.csv")
df_dia.head(2)

In [71]:
# df_dia.iloc[:, 1:4].apply(lambda x: [x.unique()]) # 시험버전
df_dia.iloc[:, 1:4].apply(lambda x: x.unique())

In [ ]:
ser_u = df_dia.iloc[:, 1:4].apply(lambda x: x.unique()).explode()
ser_repl = pd.Series(np.where(ser_u.isin(["E", "J"]), ser_u, -1), index = ser_u)
ser_repl.to_dict()

In [79]:
df_dia2 = df_dia.replace(ser_repl)
df_dia2.iloc[:, 1:4].apply(lambda x: x.unique())

cut              [-1]
color      [E, -1, J]
clarity          [-1]
dtype: object